# Integración: Generadores + Validación Estadística
**Simulación por Computador — Universidad Pedagógica y Tecnológica de Colombia**

Este notebook conecta el módulo de generadores pseudoaleatorios (Persona A)
con el módulo de validación estadística (Persona B), ejecutando las seis
pruebas estadísticas sobre las secuencias generadas por cada algoritmo.

## Requisitos
Asegurarse de que los siguientes archivos estén en la misma carpeta que este notebook:
- `generators.py` — módulo de generadores de Persona A
- `validacion_estadistica.ipynb` — módulo de pruebas de Persona B (o copiar las funciones aquí)

Ejecutar la siguiente celda solo si no están instaladas las dependencias:

In [ ]:
# Instalar dependencias necesarias (ejecutar solo una vez)
# !pip install matplotlib scipy

## 1. Importaciones

In [ ]:
import math
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm

# Importar generadores de Persona A
from generators import (
    mid_square,
    congruence,
    congruence_additive,
    congruence_multiplicative,
    general_uniform
)

# ============================================================
# MÓDULO DE VALIDACIÓN ESTADÍSTICA — Persona B
# (Copiado aquí para que el notebook sea autónomo)
# ============================================================

def prueba_medias(secuencia, nivel_confianza=0.95):
    N = len(secuencia)
    media_muestral = sum(secuencia) / N
    z = norm.ppf(1 - (1 - nivel_confianza) / 2)
    media_teorica = 0.5
    margen = z * (1 / math.sqrt(12 * N))
    limite_inferior = media_teorica - margen
    limite_superior = media_teorica + margen
    aprueba = limite_inferior <= media_muestral <= limite_superior
    return {
        "N": N, "media_muestral": round(media_muestral, 6),
        "media_teorica": media_teorica, "valor_z": round(z, 6),
        "margen": round(margen, 6), "limite_inferior": round(limite_inferior, 6),
        "limite_superior": round(limite_superior, 6), "aprueba": aprueba
    }

def prueba_varianza(secuencia, nivel_confianza=0.95):
    N = len(secuencia)
    media = sum(secuencia) / N
    varianza_muestral = sum((x - media) ** 2 for x in secuencia) / (N - 1)
    varianza_teorica = 1 / 12
    alpha = 1 - nivel_confianza
    gl = N - 1
    chi2_inferior = chi2.ppf(alpha / 2, gl)
    chi2_superior = chi2.ppf(1 - alpha / 2, gl)
    limite_inferior = chi2_inferior / (12 * gl)
    limite_superior = chi2_superior / (12 * gl)
    aprueba = limite_inferior <= varianza_muestral <= limite_superior
    return {
        "N": N, "varianza_muestral": round(varianza_muestral, 6),
        "varianza_teorica": round(varianza_teorica, 6), "alpha": round(alpha, 4),
        "grados_libertad": gl, "chi2_inferior": round(chi2_inferior, 6),
        "chi2_superior": round(chi2_superior, 6), "limite_inferior": round(limite_inferior, 6),
        "limite_superior": round(limite_superior, 6), "aprueba": aprueba
    }

def prueba_chi_cuadrado(secuencia, k=10, nivel_confianza=0.95):
    N = len(secuencia)
    frecuencia_esperada = N / k
    frecuencias_observadas = [0] * k
    for numero in secuencia:
        indice = int(numero * k)
        if indice == k:
            indice = k - 1
        frecuencias_observadas[indice] += 1
    chi2_calculado = sum(
        (obs - frecuencia_esperada) ** 2 / frecuencia_esperada
        for obs in frecuencias_observadas
    )
    gl = k - 1
    chi2_critico = chi2.ppf(nivel_confianza, gl)
    aprueba = chi2_calculado < chi2_critico
    return {
        "N": N, "k": k, "frecuencias_observadas": frecuencias_observadas,
        "frecuencia_esperada": round(frecuencia_esperada, 6),
        "chi2_calculado": round(chi2_calculado, 6),
        "chi2_critico": round(chi2_critico, 6),
        "grados_libertad": gl, "aprueba": aprueba
    }

def prueba_ks(secuencia, k=10, nivel_confianza=0.95):
    N = len(secuencia)
    frec_esperada = N / k
    frecuencias_observadas = [0] * k
    for numero in secuencia:
        indice = int(numero * k)
        if indice == k:
            indice = k - 1
        frecuencias_observadas[indice] += 1
    tabla = []
    frec_obs_acumulada = 0
    frec_esp_acumulada = 0
    for i in range(k):
        frec_obs_acumulada += frecuencias_observadas[i]
        frec_esp_acumulada += frec_esperada
        p_obs_acumulada = frec_obs_acumulada / N
        p_esp_acumulada = frec_esp_acumulada / N
        diferencia = abs(p_obs_acumulada - p_esp_acumulada)
        tabla.append({
            "intervalo": f"{round(i/k, 2)}-{round((i+1)/k, 2)}",
            "frec_observada": frecuencias_observadas[i],
            "frec_obs_acumulada": frec_obs_acumulada,
            "p_obs_acumulada": round(p_obs_acumulada, 6),
            "frec_esp_acumulada": frec_esp_acumulada,
            "p_esp_acumulada": round(p_esp_acumulada, 6),
            "diferencia": round(diferencia, 6)
        })
    dmax = max(fila["diferencia"] for fila in tabla)
    dmaxp = 1.36 / math.sqrt(N)
    aprueba = dmax < dmaxp
    return {"N": N, "k": k, "tabla": tabla,
            "dmax": round(dmax, 6), "dmaxp": round(dmaxp, 6), "aprueba": aprueba}

def prueba_poker(secuencia, nivel_confianza=0.95):
    N = len(secuencia)
    categorias = {
        "D - Todos diferentes": 0.3024, "O - Un par": 0.5040,
        "T - Dos Pares": 0.1080, "K - Tercia": 0.0720,
        "F - Full House": 0.0090, "P - Poker": 0.0045, "Q - Quintilla": 0.0001
    }
    frecuencias_observadas = {cat: 0 for cat in categorias}
    for numero in secuencia:
        digitos = str(int(round(numero * 100000))).zfill(5)[:5]
        conteo = {}
        for d in digitos:
            conteo[d] = conteo.get(d, 0) + 1
        frecuencias = sorted(conteo.values(), reverse=True)
        if frecuencias[0] == 5:
            frecuencias_observadas["Q - Quintilla"] += 1
        elif frecuencias[0] == 4:
            frecuencias_observadas["P - Poker"] += 1
        elif frecuencias[0] == 3 and frecuencias[1] == 2:
            frecuencias_observadas["F - Full House"] += 1
        elif frecuencias[0] == 3:
            frecuencias_observadas["K - Tercia"] += 1
        elif frecuencias[0] == 2 and len(frecuencias) > 1 and frecuencias[1] == 2:
            frecuencias_observadas["T - Dos Pares"] += 1
        elif frecuencias[0] == 2:
            frecuencias_observadas["O - Un par"] += 1
        else:
            frecuencias_observadas["D - Todos diferentes"] += 1
    tabla = []
    chi2_calculado = 0
    for cat, prob in categorias.items():
        obs = frecuencias_observadas[cat]
        esp = N * prob
        contribucion = (obs - esp) ** 2 / esp
        chi2_calculado += contribucion
        tabla.append({"categoria": cat, "frecuencia_observada": obs,
                      "probabilidad": prob, "frecuencia_esperada": round(esp, 4),
                      "contribucion_chi2": round(contribucion, 6)})
    chi2_critico = chi2.ppf(nivel_confianza, 6)
    aprueba = chi2_calculado < chi2_critico
    return {"N": N, "tabla": tabla, "chi2_calculado": round(chi2_calculado, 6),
            "chi2_critico": round(chi2_critico, 6), "grados_libertad": 6, "aprueba": aprueba}

def prueba_rachas(secuencia, nivel_confianza=0.95):
    N = len(secuencia)
    secuencia_ordenada = sorted(secuencia)
    if N % 2 == 0:
        mediana = (secuencia_ordenada[N//2 - 1] + secuencia_ordenada[N//2]) / 2
    else:
        mediana = secuencia_ordenada[N//2]
    signos = ["+" if x >= mediana else "-" for x in secuencia]
    n1 = signos.count("+")
    n2 = signos.count("-")
    rachas = 1
    for i in range(1, len(signos)):
        if signos[i] != signos[i-1]:
            rachas += 1
    rachas_esperadas = ((2 * n1 * n2) / N) + 1
    varianza_esperada = (2 * n1 * n2 * (2 * n1 * n2 - N)) / (N**2 * (N - 1))
    z_calculado = (rachas - rachas_esperadas) / math.sqrt(varianza_esperada)
    z_critico = norm.ppf(1 - (1 - nivel_confianza) / 2)
    aprueba = -z_critico <= z_calculado <= z_critico
    return {
        "N": N, "mediana": round(mediana, 6), "mediana_teorica": 0.5,
        "n1_positivos": n1, "n2_negativos": n2,
        "rachas_observadas": rachas, "rachas_esperadas": round(rachas_esperadas, 6),
        "varianza_esperada": round(varianza_esperada, 6),
        "z_calculado": round(z_calculado, 6), "z_critico": round(z_critico, 6),
        "rango_minimo": round(-z_critico, 6), "rango_maximo": round(z_critico, 6),
        "aprueba": aprueba
    }

print("✅ Módulos cargados correctamente.")

## 2. Función de validación completa
Esta función recibe una secuencia, ejecuta las 6 pruebas y muestra un resumen.

In [ ]:
def validar_secuencia(secuencia, nombre_generador):
    """
    Ejecuta las 6 pruebas estadísticas sobre una secuencia
    y muestra un resumen de resultados.

    Parámetros:
        secuencia: list[float] — números pseudoaleatorios entre 0 y 1
        nombre_generador: str — nombre del generador para el título
    """
    print(f"\n{'='*60}")
    print(f"  VALIDACIÓN: {nombre_generador.upper()}")
    print(f"  N = {len(secuencia)} números")
    print(f"{'='*60}")

    pruebas = [
        ("Medias",              prueba_medias(secuencia)),
        ("Varianza",            prueba_varianza(secuencia)),
        ("Chi-Cuadrado",        prueba_chi_cuadrado(secuencia)),
        ("Kolmogorov-Smirnov",  prueba_ks(secuencia)),
        ("Póker",               prueba_poker(secuencia)),
        ("Rachas",              prueba_rachas(secuencia)),
    ]

    resultados = {}
    for nombre, resultado in pruebas:
        estado = "✅ APRUEBA" if resultado["aprueba"] else "❌ FALLA"
        print(f"  {nombre:<25} {estado}")
        resultados[nombre] = resultado

    aprobadas = sum(1 for _, r in pruebas if r["aprueba"])
    print(f"\n  Resultado: {aprobadas}/6 pruebas aprobadas")
    print(f"{'='*60}\n")

    return resultados

## 3. Función de gráfico resumen
Genera un gráfico de barras mostrando qué pruebas aprobó o falló cada generador.

In [ ]:
def grafico_resumen_validacion(resultados_por_generador):
    """
    Genera un gráfico de barras comparando cuántas pruebas
    aprobó cada generador.

    Parámetros:
        resultados_por_generador: dict — {nombre_generador: dict_resultados}
    """
    generadores = list(resultados_por_generador.keys())
    pruebas = ["Medias", "Varianza", "Chi-Cuadrado",
               "Kolmogorov-Smirnov", "Póker", "Rachas"]

    fig, ax = plt.subplots(figsize=(12, 5))

    x = range(len(generadores))
    ancho = 0.12
    colores = ["steelblue", "orange", "green", "red", "purple", "brown"]

    for i, prueba in enumerate(pruebas):
        valores = []
        for gen in generadores:
            aprueba = resultados_por_generador[gen][prueba]["aprueba"]
            valores.append(1 if aprueba else 0)
        ax.bar([xi + i * ancho for xi in x], valores,
               ancho, label=prueba, color=colores[i], alpha=0.8)

    ax.set_xticks([xi + ancho * 2.5 for xi in x])
    ax.set_xticklabels(generadores, rotation=15, ha="right")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["FALLA", "APRUEBA"])
    ax.set_title("Resumen de validación por generador",
                 fontsize=14, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)

    plt.tight_layout()
    plt.savefig("grafico_resumen_validacion.png", dpi=150)
    plt.show()

## 4. Configuración de los generadores
Ajustar los parámetros según las semillas y configuraciones del equipo.

In [ ]:
# ============================================================
# PARÁMETROS DE LOS GENERADORES
# Ajustar según las semillas definidas por el equipo
# ============================================================

N = 1000  # Cantidad de números a generar por secuencia

# Cuadrados Medios
SEMILLA_MID_SQUARE = 5678

# Congruencial Mixto
XO_CONG   = 7
K_CONG    = 100
C_CONG    = 21
G_CONG    = 12

# Congruencial Aditivo (k=0)
XO_ADIT   = 5
C_ADIT    = 15
G_ADIT    = 10

# Congruencial Multiplicativo (c=0)
XO_MULT   = 3
K_MULT    = 200
G_MULT    = 14

print(f"Configuración cargada. Se generarán {N} números por secuencia.")

## 5. Generación de secuencias

In [ ]:
# Generar secuencias con cada generador de Persona A
seq_mid_square   = mid_square(SEMILLA_MID_SQUARE, N)
seq_congruencial = congruence(XO_CONG, K_CONG, C_CONG, G_CONG, N)
seq_aditivo      = congruence_additive(XO_ADIT, C_ADIT, G_ADIT, N)
seq_multiplicativo = congruence_multiplicative(XO_MULT, K_MULT, G_MULT, N)

print(f"Cuadrados Medios     → {len(seq_mid_square)} números generados")
print(f"Congruencial Mixto   → {len(seq_congruencial)} números generados")
print(f"Congruencial Aditivo → {len(seq_aditivo)} números generados")
print(f"Congruencial Mult.   → {len(seq_multiplicativo)} números generados")

# Vista previa de cada secuencia
print(f"\nVista previa (primeros 5):")
print(f"  Mid Square:    {seq_mid_square[:5]}")
print(f"  Congruencial:  {seq_congruencial[:5]}")
print(f"  Aditivo:       {seq_aditivo[:5]}")
print(f"  Multiplicativo:{seq_multiplicativo[:5]}")

## 6. Validación estadística de cada generador

In [ ]:
# Ejecutar las 6 pruebas sobre cada generador
resultados_todos = {}

resultados_todos["Cuadrados Medios"]      = validar_secuencia(seq_mid_square,    "Cuadrados Medios")
resultados_todos["Congruencial Mixto"]    = validar_secuencia(seq_congruencial,   "Congruencial Mixto")
resultados_todos["Congruencial Aditivo"]  = validar_secuencia(seq_aditivo,        "Congruencial Aditivo")
resultados_todos["Congruencial Mult."]    = validar_secuencia(seq_multiplicativo, "Congruencial Multiplicativo")

## 7. Gráfico resumen de validación

In [ ]:
grafico_resumen_validacion(resultados_todos)

## 8. Distribución de medias — múltiples simulaciones
Esta sección conecta con el Punto 2 (la rana) y el Punto 4 (EpiSim).
Recibe una lista de medias, una por cada simulación independiente,
y muestra su distribución estadística.

In [ ]:
def grafico_distribucion_medias(lista_medias, nombre="Simulaciones", nivel_confianza=0.95):
    """
    Muestra la distribución de medias calculadas a partir de
    múltiples simulaciones independientes.

    Parámetros:
        lista_medias: list[float] — media de cada simulación
        nombre: str — título descriptivo
        nivel_confianza: float — nivel de confianza (default 95%)
    """
    N = len(lista_medias)
    media_global = sum(lista_medias) / N
    media_teorica = 0.5
    z = norm.ppf(1 - (1 - nivel_confianza) / 2)
    desviacion = (sum((m - media_global)**2 for m in lista_medias) / (N - 1)) ** 0.5
    margen = z * desviacion / math.sqrt(N)
    limite_inferior = media_teorica - margen
    limite_superior = media_teorica + margen
    aprueba = limite_inferior <= media_global <= limite_superior

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.hist(lista_medias, bins=20, color="steelblue", alpha=0.7,
            edgecolor="black", label="Distribución de medias")
    ax.axvline(x=media_teorica, color="blue", linestyle="--",
               linewidth=2, label="Media teórica (0.5)")
    ax.axvline(x=media_global, color="red", linestyle="-",
               linewidth=2, label=f"Media global ({round(media_global, 4)})")
    ax.axvspan(limite_inferior, limite_superior, alpha=0.15,
               color="green", label="Intervalo de confianza 95%")

    ax.set_xlabel("Media de cada simulación", fontsize=12)
    ax.set_ylabel("Frecuencia", fontsize=12)
    ax.set_title(f"Prueba de Medias — {nombre}",
                 fontsize=14, fontweight="bold")
    ax.legend()

    texto = "APRUEBA ✓" if aprueba else "FALLA ✗"
    color_texto = "green" if aprueba else "red"
    ax.text(0.98, 0.95,
            f"Simulaciones = {N}\nMedia global = {round(media_global, 6)}\n"
            f"LI = {round(limite_inferior, 6)}\nLS = {round(limite_superior, 6)}",
            transform=ax.transAxes, fontsize=10,
            verticalalignment="top", horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    ax.text(0.02, 0.95, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto,
            verticalalignment="top", fontweight="bold")

    plt.tight_layout()
    plt.savefig(f"grafico_medias_{nombre.replace(' ', '_')}.png", dpi=150)
    plt.show()

# ============================================================
# EJEMPLO: cuando Persona C entregue las simulaciones de la rana,
# reemplazar 'medias_rana_1d' con la lista real de medias
# ============================================================

# medias_rana_1d = [mean(sim) for sim in simulaciones_1d]
# grafico_distribucion_medias(medias_rana_1d, "Caminata Aleatoria 1D")

print("Función grafico_distribucion_medias lista.")
print("Conectar con Persona C (rana) y Persona D (EpiSim) cuando tengan sus simulaciones.")